### Load Dependencies and Data

In [187]:
# Set dependencies
import pandas as pd
import numpy as np
import torch
from pyprojroot import here
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm


# Project path anchors
REPO_ROOT = here()
RAW_DATA_DIR = REPO_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = REPO_ROOT / "data" / "processed"
MODEL_DIR = REPO_ROOT / "data" / "models"

SEED = 1234


In [188]:
# Load Data

def load_split(dataset: str, split: str) -> pd.DataFrame:
    X = pd.read_csv(PROCESSED_DATA_DIR / f"{dataset}_X_{split}.csv")
    y = pd.read_csv(PROCESSED_DATA_DIR / f"{dataset}_Y_{split}.csv")
    df = pd.concat([X, y], axis=1)
    df["dataset"] = dataset
    df["split"] = split
    return df

all_data = pd.concat(
    [load_split(ds, sp) for ds in ("multiturn", "singleturn") for sp in ("train", "val", "test")],
    ignore_index=True,
)

all_data["conversation_id"] = all_data["conversation_id"].astype(str)


### Final Check for Balance, Length, and Quality

In [189]:
# Check overall structure
all_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 8148 entries, 0 to 8147
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   conversation_id  8148 non-null   str  
 1   conversation     8148 non-null   str  
 2   harm             8148 non-null   bool 
 3   dataset          8148 non-null   str  
 4   split            8148 non-null   str  
dtypes: bool(1), str(4)
memory usage: 54.5 MB


In [190]:
# check for leakage across train, val, test
all_data["conversation_id"].duplicated().sum()
all_data.duplicated(subset="conversation").sum()
all_data.groupby("conversation_id")["split"].nunique().gt(1).sum()


np.int64(1014)

In [191]:
# nulls or empty last check
all_data.isna().sum()
(all_data["conversation"].str.strip() == "").sum()

np.int64(0)

In [192]:
# Label balance look
all_data.groupby(["dataset", "split"])["harm"].mean()
all_data.groupby(["dataset", "split"])["harm"].value_counts(normalize=True).unstack()


harm                 False     True 
dataset    split                    
multiturn  test   0.499387  0.500613
           train  0.500000  0.500000
           val    0.500613  0.499387
singleturn test   0.499387  0.500613
           train  0.500000  0.500000
           val    0.499387  0.500613

In [193]:
# length and structure by source

all_data["n_chars"] = all_data["conversation"].str.len()
all_data["n_turns"] = all_data["conversation"].str.count("USER:|ASSISTANT:")
all_data.groupby("dataset")[["n_chars", "n_turns"]].describe()



n_chars                                                      \
             count          mean          std    min      25%      50%   
dataset                                                                  
multiturn   4074.0  12824.591556  6499.367922  577.0  7722.75  12108.0   
singleturn  4074.0   1046.554737  1102.429291    3.0   240.25    678.0   

                              n_turns                                     
                 75%      max   count mean  std  min  25%  50%  75%  max  
dataset                                                                   
multiturn   17495.00  35507.0  4074.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  
singleturn   1565.75  18949.0  4074.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0

In [194]:
# save the all_data file
all_data.to_parquet(PROCESSED_DATA_DIR / "all_data.parquet", index=False)


### Generate 

### Generate Embeddings

Multiturn conversations length exceeds the GPT-2 1024-token window so selected a longer context model instead.  Word2Vec doesn't capture positional information which is important in the conversations.  We selected the nomic-ai embedding model because of the length of the data and to ensure we maintain positional context.



#### GPT-2 Embeddings


In [195]:
# GPT -2 embeddings as baseline

# check to make sure the file doesn't exist
# warning that this will take a few hours to 
# create depending on processor available

out_path_gpt_2 = PROCESSED_DATA_DIR / "all_data_emb_gpt2.npy"

if not out_path_gpt_2.exists():
    # check type of processor available
    if torch.backends.mps.is_available():
        device = "mps"
    elif torch.cuda.is_available():
        device = "cuda"
    else:
        device = "cpu"

    tokenizer = AutoTokenizer.from_pretrained("gpt2")
    tokenizer.pad_token = tokenizer.eos_token

    model = AutoModel.from_pretrained("gpt2").to(device)
    model.eval()

    def embed_gpt2(texts, batch_size=8):
        all_embeddings = []
        with torch.no_grad():
            for i in tqdm(range(0, len(texts), batch_size)):
                batch = texts[i:i + batch_size]
                inputs = tokenizer(
                    batch,
                    return_tensors="pt",
                    padding=True,
                    truncation=True,
                    max_length=1024,
                ).to(device)

                outputs = model(**inputs)
                hidden_states = outputs.last_hidden_state
                mask = inputs["attention_mask"].unsqueeze(-1)

                summed = (hidden_states * mask).sum(dim=1)
                counts = mask.sum(dim=1).clamp(min=1)
                pooled = summed / counts

                all_embeddings.append(pooled.cpu().numpy())

        return np.concatenate(all_embeddings, axis=0)
    
    embeddings = embed_gpt2(all_data["conversation"].tolist())

    np.save(out_path_gpt_2, embeddings)

#### Qwen 3 Embeddings

In [196]:
# Qwen 3

# check to make sure the file doesn't exist
# warning that this will take a few hours to 
# create depending on processor available

out_path_qwen_3 = PROCESSED_DATA_DIR / "all_data_emb_qwen3.npy"

if not out_path_qwen_3.exists():
    # check type of processor available
    if torch.backends.mps.is_available():
        device = "mps"
    elif torch.cuda.is_available():
        device = "cuda"
    else:
        device = "cpu"

    model = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B", device=device)

    mask = (all_data["dataset"] == "singleturn").values

    # encode single turn first
    
    emb_single = model.encode(
        all_data.loc[mask, "conversation"].tolist(),
        batch_size=4,
        show_progress_bar=True,
        normalize_embeddings=True,
    )

    # embed multi-turn with smaller batch to avoid out of memory errors
    emb_multi = model.encode(
        all_data.loc[~mask, "conversation"].tolist(),
        batch_size=1,
        show_progress_bar=True,
        normalize_embeddings=True,
    )

    embeddings = np.empty((len(all_data), emb_single.shape[1]), dtype=emb_single.dtype)
    embeddings[mask] = emb_single
    embeddings[~mask] = emb_multi

    np.save(out_path_qwen_3, embeddings)

#### Nomic-AI Embeddings

In [197]:
out_path = PROCESSED_DATA_DIR / "all_data_emb_nomic.npy"

# check to make sure the file doesn't exist
# warning that this will take a few hours to 
# create depending on processor available

if not out_path.exists():
    # check type of processor available
    if torch.backends.mps.is_available():
        device = "mps"
    elif torch.cuda.is_available():
        device = "cuda"
    else:
        device = "cpu"

    # create the model and fix the max input length to the models max
    model = SentenceTransformer("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True, device=device)
    model.max_seq_length = 8192

    mask = (all_data["dataset"] == "singleturn").values

    # encode the single turn data with a larger batch size for speed 
    emb_single = model.encode(
        ("classification: " + all_data.loc[mask, "conversation"]).tolist(),
        batch_size=8,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    # encode the multiturn with smaller batch size
    emb_multi = model.encode(
        ("classification: " + all_data.loc[~mask, "conversation"]).tolist(),
        batch_size=1,
        show_progress_bar=True,
        normalize_embeddings=True,
    )

    embeddings = np.empty((len(all_data), emb_single.shape[1]), dtype=emb_single.dtype)
    embeddings[mask] = emb_single
    embeddings[~mask] = emb_multi

    # save embeddings as np.array file
    np.save(out_path, embeddings)



### Review and check the embeddings

In [198]:
# load embeddings
emb_gpt2 = np.load(PROCESSED_DATA_DIR / "all_data_emb_gpt2.npy")
emb_qwen3 = np.load(PROCESSED_DATA_DIR / "all_data_emb_qwen3.npy")
emb_nomic = np.load(PROCESSED_DATA_DIR / "all_data_emb_nomic.npy")

In [199]:
# Shape of the embeddings
print("--- Embeddings for the Multi-Turn Data ---\n")
print(f"Size of the original data: {len(all_data)}")
print(f"Shape of the GPT2 embeddings: {emb_gpt2.shape}")
print(f"Shape of the QWEN3 embeddings: {emb_qwen3.shape}")
print(f"Shape of the Nomic embeddings: {emb_nomic.shape}")

--- Embeddings for the Multi-Turn Data ---

Size of the original data: 8148
Shape of the GPT2 embeddings: (8148, 768)
Shape of the QWEN3 embeddings: (8148, 1024)
Shape of the Nomic embeddings: (8148, 768)


### WildGuardMix Single Turn Data

In [200]:
# Load Data
train_raw = RAW_DATA_DIR / "wildguardmix" / "train" / "wildguard_train.parquet"
test_raw = RAW_DATA_DIR / "wildguardmix" / "test" / "wildguard_test.parquet"

df_train = pd.read_parquet(train_raw)
df_test = pd.read_parquet(test_raw)

print("raw train:", df_train.shape)
print("raw test :", df_test.shape)
df_train.head(3)

raw train: (86759, 7)
raw test : (1725, 10)


,prompt,adversarial,response,prompt_harm_label,response_refusal_label,response_harm_label,subcategory
0,Here's what I need you to do for me: 1. Transl...,True,"I can help with some of your requests, but I h...",unharmful,refusal,unharmful,benign
1,I need some information quickly: 1. What is th...,True,I can provide answers to points 1 through 3 of...,unharmful,refusal,unharmful,benign
2,Please do the following tasks: 1. Explain what...,True,I'll gladly assist you with your queries: 1. S...,unharmful,refusal,unharmful,benign


#### Build the single turn task

In [201]:
def build_singleturn(df):
    df = df.dropna(subset=["prompt_harm_label"]).drop_duplicates(subset="prompt").copy()
    df["y"] = (df["prompt_harm_label"] == "harmful").astype(int)
    return df

train_df = build_singleturn(df_train)
test_df = build_singleturn(df_test)

# Carve validation out of the (deduped) training partition; test stays locked.
X_train, X_val, y_train, y_val = train_test_split(
    train_df["prompt"], train_df["y"],
    test_size=0.15, random_state=SEED, stratify=train_df["y"],
)
X_test, y_test = test_df["prompt"], test_df["y"]

print(f"train = {len(X_train):>6}   val = {len(X_val):>6}   test = {len(X_test):>6}")

train =  40674   val =   7178   test =   1699


In [202]:
wgm_aligned = pd.concat([
    pd.DataFrame({"conversation": X_train, "harm": y_train.astype(bool), "split": "train"}),
    pd.DataFrame({"conversation": X_val,   "harm": y_val.astype(bool),   "split": "val"}),
    pd.DataFrame({"conversation": X_test,  "harm": y_test.astype(bool), "split": "test"}),
], ignore_index=True)
wgm_aligned["conversation_id"] = wgm_aligned.index.astype(str)

print(wgm_aligned["split"].value_counts())
print(wgm_aligned.groupby("split")["harm"].mean())

wgm_aligned.to_parquet(PROCESSED_DATA_DIR / "wildguardmix_aligned.parquet", index=False)


split
train    40674
val       7178
test      1699
Name: count, dtype: int64
split
test     0.443790
train    0.517579
val      0.517554
Name: harm, dtype: float64


#### Generate Embeddings for single turn

#### GPT-2

In [203]:
out_path = PROCESSED_DATA_DIR / "wildguardmix_aligned_emb_gpt2.npy"

if not out_path.exists():
    if torch.backends.mps.is_available():
        device = "mps"
    elif torch.cuda.is_available():
        device = "cuda"
    else:
        device = "cpu"

    tokenizer = AutoTokenizer.from_pretrained("gpt2")
    tokenizer.pad_token = tokenizer.eos_token
    model = AutoModel.from_pretrained("gpt2").to(device)
    model.eval()

    all_embeddings = []
    with torch.no_grad():
        for i in tqdm(range(0, len(wgm_aligned), 16)):
            batch = wgm_aligned["conversation"].iloc[i:i + 16].tolist()
            inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=1024).to(device)
            outputs = model(**inputs)
            mask = inputs["attention_mask"].unsqueeze(-1)
            pooled = (outputs.last_hidden_state * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
            all_embeddings.append(pooled.cpu().numpy())

    np.save(out_path, np.concatenate(all_embeddings, axis=0))


  0%|          | 14/3097 [00:16<1:02:09,  1.21s/it]


KeyboardInterrupt: 

#### Qwen-3

In [ ]:
out_path = PROCESSED_DATA_DIR / "wildguardmix_aligned_emb_qwen3.npy"

if not out_path.exists():
    if torch.backends.mps.is_available():
        device = "mps"
    elif torch.cuda.is_available():
        device = "cuda"
    else:
        device = "cpu"

    model = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B", device=device)
    embeddings = model.encode(
        wgm_aligned["conversation"].tolist(),
        batch_size=32,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    np.save(out_path, embeddings)


#### Nomic

In [ ]:
out_path = PROCESSED_DATA_DIR / "wildguardmix_aligned_emb_nomic.npy"

if not out_path.exists():
    if torch.backends.mps.is_available():
        device = "mps"
    elif torch.cuda.is_available():
        device = "cuda"
    else:
        device = "cpu"

    model = SentenceTransformer("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True, device=device)
    model.max_seq_length = 8192
    embeddings = model.encode(
        ("classification: " + wgm_aligned["conversation"]).tolist(),
        batch_size=32,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    np.save(out_path, embeddings)


#### Review and Check

In [ ]:
emb_gpt2 = np.load(PROCESSED_DATA_DIR / "wildguardmix_aligned_emb_gpt2.npy")
emb_qwen3 = np.load(PROCESSED_DATA_DIR / "wildguardmix_aligned_emb_qwen3.npy")
emb_nomic = np.load(PROCESSED_DATA_DIR / "wildguardmix_aligned_emb_nomic.npy")

print(f"rows in wgm_aligned: {len(wgm_aligned)}")
print(f"gpt2:  {emb_gpt2.shape}")
print(f"qwen3: {emb_qwen3.shape}")
print(f"nomic: {emb_nomic.shape}")
